<a href="https://colab.research.google.com/github/ekonjmrivas-devops/llm_engineering/blob/mis-ejercicios/W3_PRACTICA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Semana 3 - Práctica


**Bloque - Librerías**

In [1]:
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 960.7/960.7 kB 18.9 MB/s eta 0:00:00


In [2]:
!pip install -q -U bitsandbytes>=0.46.1

In [4]:
import os
import requests
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from google.colab import drive
from huggingface_hub import login
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch
from datetime import datetime
import anthropic

**Bloque - Setup y conexión a Drive**

In [ ]:
# Conexión a Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#drive.flush_and_unmount()
#drive.mount('/content/drive', force_remount=True)

In [ ]:
# Rutas fijas del proyecto
BASE_PATH = '/content/drive/MyDrive/cursollms/week3'
INPUT_PATH = f'{BASE_PATH}/Inbound files'
OUTPUT_PATH = f'{BASE_PATH}/Outbound files'

In [ ]:
# Crear carpetas si no existen
os.makedirs(INPUT_PATH, exist_ok=True)
os.makedirs(OUTPUT_PATH, exist_ok=True)

In [ ]:
print(f"Carpeta de entrada: {INPUT_PATH}")
print(f"Carpeta de salida: {OUTPUT_PATH}")

Carpeta de entrada: /content/drive/MyDrive/cursollms/week3/Inbound files
Carpeta de salida: /content/drive/MyDrive/cursollms/week3/Outbound files


**Bloque - Listar archivos de la carpeta de entrada**

In [ ]:
# Listar archivos disponibles en directorio de Drive

def listar_archivos_origen():
    """
    Lista los archivos disponibles en la carpeta de entrada de Drive.
    Devuelve una lista de nombres de archivo (sin la ruta completa),
    pensada para poblar el Dropdown de Gradio.
    """
    extensiones_validas = ('.mp3', '.wav', '.m4a', '.txt')

    archivos = [
        f for f in os.listdir(INPUT_PATH)
        if f.lower().endswith(extensiones_validas)
    ]

    archivos.sort()
    return archivos

In [ ]:
# Comprobación de la función
archivos_disponibles = listar_archivos_origen()
print(f"Archivos encontrados: {len(archivos_disponibles)}")
for a in archivos_disponibles:
    print(f"  - {a}")

Archivos encontrados: 2
  - Desarrollo de un caso práctico.txt
  - denver_extract.mp3


In [ ]:
# Diagnóstico: ver TODO lo que hay en la carpeta, sin filtrar por extensión
print(os.listdir(INPUT_PATH))

['Irregular Verbs.pdf', 'training-5-azure-devops-pipelines.md', 'Desarrollo de un caso práctico.txt', 'denver_extract.mp3']


**Bloque - Transcripción de audio con Whisper (OpenAI API)**

In [ ]:
# Crear cliente de OpenAI
client = OpenAI(api_key= userdata.get('OPENAI_API_KEY'))
AUDIO_MODEL = "whisper-1"

In [ ]:
def transcribir_audio(nombre_archivo):
    """
    Transcribe un archivo de audio a texto usando Whisper (OpenAI API).
    nombre_archivo: nombre del archivo dentro de INPUT_PATH (ej. 'denver_extract.mp3')
    Devuelve: el texto transcrito.
    """
    ruta_completa = os.path.join(INPUT_PATH, nombre_archivo)

    with open(ruta_completa, "rb") as audio_file:
        transcripcion = client.audio.transcriptions.create(
            model=AUDIO_MODEL,
            file=audio_file,
            response_format="text"
        )

    return transcripcion

In [ ]:
texto_transcrito = transcribir_audio("denver_extract.mp3")
print(texto_transcrito[:500])
print(f"\nLongitud total: {len(texto_transcrito)} caracteres")

and kind of the confluence of this whole idea of the confluence week, the merging of two rivers and as we've kind of seen recently in politics and in the world, there's a lot of situations where water is very important right now and it's a very big issue. So that is the reason that the back of the logo is considered water. So let me see the creation of the logo here. So that basically kind of sums up the reason behind the logo and all the meanings behind the symbolism and you'll hear a little bi

Longitud total: 12521 caracteres


**Bloque - Lectura de archivo de texto**

In [ ]:
def leer_texto(nombre_archivo):
    """
    Lee directamente un archivo de texto (.txt) de la carpeta de entrada.
    nombre_archivo: nombre del archivo dentro de INPUT_PATH (ej. 'Desarrollo de un caso práctico.txt')
    Devuelve: el contenido del archivo como string.
    """
    ruta_completa = os.path.join(INPUT_PATH, nombre_archivo)

    with open(ruta_completa, "r", encoding="utf-8") as f:
        contenido = f.read()

    return contenido

In [ ]:
texto_leido = leer_texto("Desarrollo de un caso práctico.txt")
print(texto_leido[:500])
print(f"\nLongitud total: {len(texto_leido)} caracteres")

Apertura: domingo, 5 de abril de 2026, 00:00
Cierre: martes, 21 de abril de 2026, 00:00
Se pide desarrollar dos ejercicios (uno de terraform y otro de ansible) para comprobar si se han entendido de forma correcta los conceptos explicados en el módulo

Ejercicios_Modulo4_IaC.pdf Ejercicios_Modulo4_IaC.pdf21 de marzo de 2026, 13:38


Longitud total: 332 caracteres


**Bloque - Seleccionar tipo de archivo de entrada**

In [ ]:
def obtener_texto_origen(nombre_archivo, tipo_origen):
    """
    Obtiene el texto de origen, ya sea transcribiendo audio o leyendo texto directo.
    tipo_origen: "audio" o "texto"
    """
    if tipo_origen == "audio":
        return transcribir_audio(nombre_archivo)
    elif tipo_origen == "texto":
        return leer_texto(nombre_archivo)
    else:
        raise ValueError(f"Tipo de origen no reconocido: {tipo_origen}")

In [ ]:
texto_prueba = obtener_texto_origen("Desarrollo de un caso práctico.txt", "texto")
print(texto_prueba[:300])

Apertura: domingo, 5 de abril de 2026, 00:00
Cierre: martes, 21 de abril de 2026, 00:00
Se pide desarrollar dos ejercicios (uno de terraform y otro de ansible) para comprobar si se han entendido de forma correcta los conceptos explicados en el módulo

Ejercicios_Modulo4_IaC.pdf Ejercicios_Modulo4_Ia


**Bloque - Configuración de cuantización y carga de Llama local**

In [ ]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
# Carga el modelo de Llama solo cuando sea necesario

LLAMA = "meta-llama/Meta-Llama-3.1-8B-Instruct"

llama_tokenizer = None
llama_model = None

def cargar_llama_si_necesario():
    """
    Carga el modelo Llama solo si no está ya cargado en memoria.
    Evita recargarlo innecesariamente entre llamadas.
    """
    global llama_tokenizer, llama_model

    if llama_model is None:
        print("Cargando Llama por primera vez...")
        llama_tokenizer = AutoTokenizer.from_pretrained(LLAMA)
        llama_tokenizer.pad_token = llama_tokenizer.eos_token
        llama_model = AutoModelForCausalLM.from_pretrained(
            LLAMA, device_map="auto", quantization_config=quant_config
        )
        print("Llama cargado correctamente.")
    else:
        print("Llama ya estaba cargado en memoria.")

**Bloque - Generación de resumen y acciones**

In [ ]:
# Configuración Claude

CLAUDE = "claude-sonnet-4-6"

claude_client = anthropic.Anthropic(
    api_key=userdata.get('ANTHROPIC_API_KEY')
)

In [ ]:
# Generar resumen con Claude

def generar_con_claude(texto):
    """
    Genera resumen y acciones usando Claude API.
    """
    messages = [
        {
            "role": "user",
            "content": f"""Eres un asistente experto en redacción de actas de reuniones.

Dado el siguiente texto de una reunión, genera:
1. Un resumen ejecutivo del acta (máximo 300 palabras)
2. Lista de acciones acordadas o próximos pasos

Texto de la reunión:
{texto}

Responde en español, con este formato exacto:
## Resumen ejecutivo
[resumen aquí]

## Acciones acordadas
- [acción 1]
- [acción 2]
..."""
        }
    ]

    response = claude_client.messages.create(
        model=CLAUDE,
        max_tokens=1000,
        messages=messages
    )

    return response.content[0].text


In [ ]:
# Generar resumen con Llama

def generar_con_llama(texto):
    """
    Genera resumen y acciones usando Llama local.
    """
    cargar_llama_si_necesario()

    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
Eres un asistente experto en redacción de actas de reuniones. Responde siempre en español.<|eot_id|>
<|start_header_id|>user<|end_header_id|>
Dado el siguiente texto de una reunión, genera:
1. Un resumen ejecutivo del acta (máximo 300 palabras)
2. Lista de acciones acordadas o próximos pasos

Texto:
{texto}

Responde con este formato exacto:
## Resumen ejecutivo
[resumen aquí]

## Acciones acordadas
- [acción 1]
- [acción 2]<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>"""

    inputs = llama_tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    outputs = llama_model.generate(
        **inputs,
        max_new_tokens=1000
    )

    return llama_tokenizer.decode(outputs[0], skip_special_tokens=True)

**Bloque - Función orquestadora del modelo para generar resumen**

In [ ]:
# Función orquestadora para generar resumen en función del modelo

def generar_resumen(texto, modelo="claude"):
    """
    Genera resumen y acciones con el modelo seleccionado.
    modelo: "claude" o "llama"
    """
    if modelo == "claude":
        return generar_con_claude(texto)
    elif modelo == "llama":
        return generar_con_llama(texto)
    else:
        raise ValueError(f"Modelo no reconocido: {modelo}")

In [ ]:
resultado = generar_resumen(texto_prueba, modelo="llama")
print(resultado)

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Llama ya estaba cargado en memoria.


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


system
Eres un asistente experto en redacción de actas de reuniones. Responde siempre en español.
user
Dado el siguiente texto de una reunión, genera:
1. Un resumen ejecutivo del acta (máximo 300 palabras)
2. Lista de acciones acordadas o próximos pasos

Texto:
Apertura: domingo, 5 de abril de 2026, 00:00
Cierre: martes, 21 de abril de 2026, 00:00
Se pide desarrollar dos ejercicios (uno de terraform y otro de ansible) para comprobar si se han entendido de forma correcta los conceptos explicados en el módulo

Ejercicios_Modulo4_IaC.pdf Ejercicios_Modulo4_IaC.pdf21 de marzo de 2026, 13:38


Responde con este formato exacto:
## Resumen ejecutivo
[resumen aquí]

## Acciones acordadas
- [acción 1]
- [acción 2]
assistant

## Resumen ejecutivo
En la reunión celebrada del domingo 5 de abril de 2026 al martes 21 de abril de 2026, se abordó el desarrollo de dos ejercicios para comprobar la comprensión de los conceptos explicados en el módulo. Los ejercicios tenían como objetivo evaluar la capaci

**Bloque - Generación automática del nombre del archivo de salida**

In [3]:
def generar_nombre_archivo(nombre_origen, extension="txt"):
    """
    Genera un nombre de archivo de salida basado en:
    - nombre del archivo de origen (sin extensión)
    - fecha actual (YYYYMMDD)
    - hora actual (HHMMSS)
    - extensión elegida (txt o pdf)

    Ejemplo: denver_extract_20260624_143052.txt
    """
    nombre_base = os.path.splitext(nombre_origen)[0]
    ahora = datetime.now()
    timestamp = ahora.strftime("%Y%m%d_%H%M%S")

    return f"{nombre_base}_{timestamp}.{extension}"

Procesamiento con LLM
- Resumen del acta/minuta
- Extracción de acciones/próximos pasos

Generación del archivo de salida
- Nombre automático
- Formato TXT o PDF
- Guardado en Drive

Interfaz Gradio
- Formulario de inputs
- Preview de resultado (scroll)
- Previo del archivo de salida
